<a href="https://colab.research.google.com/github/lab-rasool/SIIM/blob/main/notebooks/SIIM_LocalLLMs_Backup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIIM 2026 Learning Lab — Backup Cloud Instance
### Running Local LLMs Behind Institutional Firewalls (LL4022)

A **fallback** for the hands-on labs: if a laptop won't cooperate, run these
cells top to bottom to get Ollama serving an open model (Lab 1), a private
ChatGPT-style interface at a public URL (Lab 2), and the Lab 3 clinical
workflows — radiology summarization and pathology extraction.

**How to use this notebook**
- Run the cells **in order, top to bottom** the first time.
- ▶️ Most cells just need a **click** — wait for them to finish, then move on.
- ✏️ Cells marked **EDIT ME** have a few settings at the top you can change.
- It's faster on a GPU: **Runtime → Change runtime type → T4 GPU**.

> ## ⚠️ Read this first — synthetic data only
> A Colab instance is a **public cloud machine** — the *opposite* of "behind the
> firewall." It's the right tool for a conference demo with **synthetic data**,
> and the wrong tool for anything real.
>
> **Never paste real patient data or PHI** into this notebook or the public URL
> it creates. The public link is unauthenticated until you make an OpenWebUI
> account in the browser. When you're done, shut everything down with
> **Runtime → Disconnect and delete runtime**.

Modeled on [Axenide/Open-WebUI-Colab](https://github.com/Axenide/Open-WebUI-Colab).


## Step 0 — Pick your model(s)  ✏️
`llama3.2` matches Lab 1 (~2 GB, fast). Set `LARGE_MODEL` to also pull a bigger
model for the side-by-side comparison in Step 5 — or leave it blank to skip.


In [ ]:
# ✏️ Choose your model(s), then run this cell.
SMALL_MODEL = "llama3.2"      # Lab 1 default (~2 GB)
LARGE_MODEL = "qwen2.5:7b"    # for the small-vs-large comparison; "" to skip

import os
os.environ["SMALL_MODEL"] = SMALL_MODEL
os.environ["LARGE_MODEL"] = LARGE_MODEL

# Ollama speed/memory settings (every server we start later inherits these):
#   flash attention + q8_0 KV cache  -> longer reports fit on a T4
#   keep-alive -1                    -> keep the model loaded between requests
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"]   = "q8_0"
os.environ["OLLAMA_KEEP_ALIVE"]      = "-1"

print("Will pull:", SMALL_MODEL, "and", LARGE_MODEL or "(no large model)")


## Step 1 — Install Ollama and pull the model(s)  ▶️
Installs the model runtime, starts it, and downloads your model(s). About two
minutes on a T4 GPU. On CPU, `llama3.2` works but is slow (and a 7B model isn't
practical for a live demo — set `LARGE_MODEL = ""` in Step 0).


In [ ]:
# Installs the Ollama runtime, starts it, and pulls your model(s).

# Ollama needs zstd to unpack, and pciutils/lshw to detect the GPU. A fresh
# Colab runtime has none of these, so install them first (without them it
# silently falls back to CPU even on a GPU runtime).
!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd pciutils lshw

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import shutil, subprocess, time, os
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(ollama_bin), 'Ollama did not install - re-run this cell.'

# Show the GPU. If this errors you're on CPU: Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU detected - running on CPU.'

# Colab has no systemd, so start the server ourselves (inherits the Step 0 settings).
subprocess.Popen([ollama_bin, 'serve'])
time.sleep(8)

# Download the model(s)
!ollama pull $SMALL_MODEL
if os.environ.get("LARGE_MODEL"):
    get_ipython().system('ollama pull $LARGE_MODEL')

# Show what's installed locally
!curl -s localhost:11434/api/tags | python3 -m json.tool


## Step 2 — Install OpenWebUI  ▶️
Sets up the private, ChatGPT-style chat interface (using
[uv](https://docs.astral.sh/uv/) for a fast, isolated Python 3.11 environment).
This only **installs** it — we start it in Step 4.


In [ ]:
# Installs OpenWebUI into its own Python 3.11 environment with uv.
!pip install -q uv

# uv fetches a managed Python 3.11 and builds the environment at /content/venv
# (absolute path, so later cells always find it).
!export UV_VENV_CLEAR=1
!uv venv --python 3.11 /content/venv
!uv pip install --python /content/venv/bin/python open-webui -q
print("OpenWebUI installed at /content/venv")


## Step 3 — Get the lab files  ▶️
Clones the workshop repo: the Lab 3 scripts and the **synthetic** clinical
datasets.


In [ ]:
# Brings in the Lab 3 scripts and the SYNTHETIC datasets.
import os, glob

REPO_DIR = "/content/SIIM"
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !rm -rf {REPO_DIR}
    !git clone https://github.com/lab-rasool/SIIM.git {REPO_DIR}

# Find where the lab scripts live and remember it for later cells.
hits = glob.glob(os.path.join(REPO_DIR, '**', 'summarize_report.py'), recursive=True)
LAB_DIR = os.path.dirname(hits[0]) if hits else REPO_DIR
os.environ['LAB_DIR'] = LAB_DIR
%cd {LAB_DIR}
print('Lab scripts:', LAB_DIR)
!ls -1


## Step 4 — Open the chat interface (public link)  ▶️
Starts everything and prints a public link (via [Pinggy](https://pinggy.io/)).
Open it, click through Pinggy's one-time "Enter Site" splash, sign up with any
email/password (it stays on this runtime), and pick your model from the dropdown
at the top-left.

> **No link?** Just run the cell again. The free tunnel lasts 60 minutes and the
> URL changes each time you start a new one — re-run for a fresh link.


In [ ]:
# Makes OpenWebUI work end-to-end: starts Ollama, pulls the model, starts
# OpenWebUI pointed at Ollama, and opens a clean public link via Pinggy.
# Click to run. If no link appears, just run it again. No typing required.
import os, re, json, time, shutil, subprocess, urllib.request

def _up(url):
    try: urllib.request.urlopen(url, timeout=3); return True
    except Exception: return False

# 1) Make sure Ollama is running, then pull the model so OpenWebUI has something.
#    (serve inherits the OLLAMA_* perf env set in Step 0.)
ob = shutil.which("ollama") or "/usr/local/bin/ollama"
if not _up("http://localhost:11434/api/tags") and os.path.exists(ob):
    subprocess.Popen([ob, "serve"])
    for _ in range(20):
        if _up("http://localhost:11434/api/tags"): break
        time.sleep(1)
MODEL = os.environ.get("SMALL_MODEL", "llama3.2")
if _up("http://localhost:11434/api/tags"):
    subprocess.run(["ollama", "pull", MODEL])

# 2) Start OpenWebUI, explicitly pointed at the local Ollama. These two env
#    vars are what make your pulled models usable *inside* OpenWebUI: OpenWebUI
#    talks to Ollama server-side over localhost, so it works the same whether
#    you reach the UI locally or through the public Pinggy link.
if not os.path.exists("/content/venv/bin/open-webui"):
    print("\u26a0\ufe0f OpenWebUI isn't installed yet - run Step 2 first, then this cell.")
else:
    if not _up("http://localhost:8081"):
        env = dict(os.environ,
                   OLLAMA_BASE_URL="http://127.0.0.1:11434",
                   ENABLE_OLLAMA_API="true")
        subprocess.Popen(["/content/venv/bin/open-webui", "serve", "--port", "8081"],
                         env=env, stdout=open("/content/openwebui.log","w"),
                         stderr=subprocess.STDOUT)
        print("Starting OpenWebUI\u2026 first boot can take ~30-60s.")
        for _ in range(60):
            if _up("http://localhost:8081"): break
            time.sleep(2)

    # 3) Public link via Pinggy. Open a reverse tunnel for the UI (8081), plus a
    #    LOCAL forward to Pinggy's URL API on 4300 so we can read the real link
    #    as clean JSON instead of scraping the banner. Ollama (11434) stays
    #    private; only the UI is exposed. Free tier: random URL, 60-min sessions.
    get_ipython().system('pkill -f "a.pinggy.io" 2>/dev/null')
    time.sleep(1)
    subprocess.Popen(
        ["ssh", "-p", "443",
         "-o", "StrictHostKeyChecking=no",
         "-o", "UserKnownHostsFile=/dev/null",
         "-o", "ServerAliveInterval=30",
         "-o", "ServerAliveCountMax=3",
         "-R", "0:localhost:8081",
         "-L", "4300:localhost:4300",
         "free@a.pinggy.io"],
        stdin=subprocess.DEVNULL,
        stdout=open("/content/pinggy.log", "w"), stderr=subprocess.STDOUT)

    # The real tunnel URL ends in pinggy.link (or pinggy-free.link). The
    # dashboard link (dashboard.pinggy.io) is shown in the banner but is NOT the
    # tunnel - only accept *.link hosts so we never grab the dashboard by mistake.
    LINK_RE = re.compile(r"https://[a-z0-9.\-]+\.pinggy[a-z0-9.\-]*\.link", re.I)
    url = None
    for _ in range(40):
        # Preferred: Pinggy's local URL API returns the links as clean JSON.
        try:
            data = json.load(urllib.request.urlopen("http://localhost:4300/urls", timeout=3))
            for u in data.get("urls", []):
                if u.startswith("https://"): url = u; break
            if url: break
        except Exception:
            pass
        # Fallback: scrape the SSH log, but only for a real *.link URL.
        try: log = open("/content/pinggy.log").read()
        except FileNotFoundError: log = ""
        m = LINK_RE.findall(log)
        if m: url = m[0]; break
        time.sleep(2)

    if url and _up("http://localhost:8081"):
        print("\n\u2705 OpenWebUI is ready - open this link:\n   " + url)
        print("   (Pinggy shows a one-time 'Enter Site' splash - click through it.)")
        print("   Sign up with any email/password on first visit (stays on this runtime).")
        print("   Your pulled model appears in the dropdown (top-left). Refresh if not.")
    elif url:
        print("\nLink is up but OpenWebUI is still booting - open it in ~30s:\n   " + url)
    else:
        print("\nNo public link yet - just run this cell again.")
        try:
            tail = open("/content/pinggy.log").read().strip().splitlines()[-5:]
            if tail: print("   (last tunnel messages: " + " | ".join(tail) + ")")
        except Exception:
            pass


## Step 5 — Run the Lab 3 clinical workflows  ▶️ ✏️
The two tasks from Lab 3, on synthetic reports: a 3-line **radiology** impression
and structured **pathology** JSON — plus a small-vs-larger model comparison. Run
them in order. Each cell has an **EDIT ME** block at the top, so you can switch
the model or point it at a different synthetic report.

Reports you can use:
- **Radiology:** `ct_chest_001.txt` · `ct_abdomen_002.txt` · `mri_brain_003.txt` · `xray_chest_004.txt`
- **Pathology:** `path_breast_001.txt` · `path_lung_002.txt` · `path_colon_003.txt` · `path_prostate_004.txt`


In [ ]:
# ✏️ EDIT ME ─────────────────────────────────────────────────────
MODEL  = "llama3.2"                          # model to use (from Step 0)
REPORT = "data/radiology/ct_chest_001.txt"   # which synthetic report
# ─────────────────────────────────────────────────────────────────

# plumbing: make sure the local model is awake (safe to ignore) ----
import os, time, shutil, subprocess, urllib.request
LAB = os.environ.get("LAB_DIR", "/content/SIIM")
def _ollama_ready():
    try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3); return True
    except Exception: return False
if not _ollama_ready():
    subprocess.Popen([shutil.which("ollama") or "/usr/local/bin/ollama", "serve"])
    for _ in range(20):
        if _ollama_ready(): break
        time.sleep(1)
# ------------------------------------------------------------------

# Radiology summarization -> 3-line structured impression
if _ollama_ready():
    subprocess.run(["ollama", "pull", MODEL])   # instant if already downloaded
    get_ipython().system(f'python3 "{LAB}/summarize_report.py" --model {MODEL} --report "{LAB}/{REPORT}"')
else:
    print("⚠️  Ollama isn't ready - run Step 1 (Install Ollama) first, then this cell.")


In [ ]:
# ✏️ EDIT ME ─────────────────────────────────────────────────────
MODEL  = "llama3.2"                           # model to use (from Step 0)
REPORT = "data/pathology/path_lung_002.txt"   # which synthetic report
# ─────────────────────────────────────────────────────────────────

# plumbing: make sure the local model is awake (safe to ignore) ----
import os, time, shutil, subprocess, urllib.request
LAB = os.environ.get("LAB_DIR", "/content/SIIM")
def _ollama_ready():
    try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3); return True
    except Exception: return False
if not _ollama_ready():
    subprocess.Popen([shutil.which("ollama") or "/usr/local/bin/ollama", "serve"])
    for _ in range(20):
        if _ollama_ready(): break
        time.sleep(1)
# ------------------------------------------------------------------

# Pathology extraction -> structured, validated JSON
if _ollama_ready():
    subprocess.run(["ollama", "pull", MODEL])
    get_ipython().system(f'python3 "{LAB}/extract_pathology.py" --model {MODEL} --report "{LAB}/{REPORT}"')
else:
    print("⚠️  Ollama isn't ready - run Step 1 (Install Ollama) first, then this cell.")


In [ ]:
# ✏️ EDIT ME ─────────────────────────────────────────────────────
SMALL  = "llama3.2"                           # small model
LARGE  = "qwen2.5:7b"                          # larger model (pull it in Step 0); "" to skip
REPORT = "data/radiology/mri_brain_003.txt"    # which synthetic report
# ─────────────────────────────────────────────────────────────────

# plumbing: make sure the local model is awake (safe to ignore) ----
import os, time, shutil, subprocess, urllib.request
LAB = os.environ.get("LAB_DIR", "/content/SIIM")
def _ollama_ready():
    try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3); return True
    except Exception: return False
if not _ollama_ready():
    subprocess.Popen([shutil.which("ollama") or "/usr/local/bin/ollama", "serve"])
    for _ in range(20):
        if _ollama_ready(): break
        time.sleep(1)
# ------------------------------------------------------------------

# Small vs. larger model, side by side
if not LARGE:
    print('Set LARGE to a model you pulled in Step 0 (e.g. "qwen2.5:7b") to compare.')
elif _ollama_ready():
    subprocess.run(["ollama", "pull", SMALL])
    subprocess.run(["ollama", "pull", LARGE])
    get_ipython().system(f'python3 "{LAB}/summarize_report.py" --compare {SMALL} {LARGE} --report "{LAB}/{REPORT}"')
else:
    print("⚠️  Ollama isn't ready - run Step 1 (Install Ollama) first, then this cell.")


## Step 6 — Your turn: a code playground  ✏️
Step 4 let you experiment in the browser. This is the same idea **in code**:
drop in any synthetic report, pick a model, choose the task, and run. It reuses
the exact Lab 3 logic, so the output matches Step 5 — only **your input and
model** change. Re-run as often as you like; everything stays on this machine.


In [ ]:
# ════════════════════════════════════════════════════════════════
#  PLAYGROUND  ·  edit the three settings, then run this cell
#  Synthetic data only — nothing real goes in.
# ════════════════════════════════════════════════════════════════

MODEL = "llama3.2"          # any model you pulled in Step 0 (e.g. "qwen2.5:7b")

TASK  = "radiology"         # "radiology" -> 3-line impression
                            # "pathology" -> structured JSON

REPORT = """
SYNTHETIC CT CHEST - replace this with your own synthetic text.
FINDINGS: 8 mm nodule in the right lower lobe, new since the prior study.
No pleural effusion. No lymphadenopathy.
IMPRESSION: New 8 mm right lower lobe nodule; short-interval CT recommended.
"""

# ── plumbing: load the Lab 3 logic + wake the model (safe to ignore) ──
import os, sys, time, json, shutil, subprocess, urllib.request
LAB = os.environ.get("LAB_DIR", "/content/SIIM")
sys.path.insert(0, LAB)
from utils.ollama_client import DEFAULT_HOST   # local Ollama, no data leaves the box
import summarize_report as radiology
import extract_pathology as pathology

def _ready():
    try: urllib.request.urlopen(DEFAULT_HOST + "/api/tags", timeout=3); return True
    except Exception: return False
if not _ready():
    subprocess.Popen([shutil.which("ollama") or "/usr/local/bin/ollama", "serve"])
    for _ in range(20):
        if _ready(): break
        time.sleep(1)
if _ready():
    subprocess.run(["ollama", "pull", MODEL])   # instant if already downloaded
# ──────────────────────────────────────────────────────────────────────

if not _ready():
    print("⚠️  Ollama isn't ready - run Step 1 (Install Ollama) first, then this cell.")
elif TASK == "radiology":
    print(radiology.summarize(REPORT, MODEL, DEFAULT_HOST))
elif TASK == "pathology":
    obj, raw = pathology.extract(REPORT, MODEL, DEFAULT_HOST)
    print(json.dumps(obj, indent=2) if obj else raw)
else:
    print('Set TASK to "radiology" or "pathology".')


**Go further** — the cell below is the raw building block the labs are built on.
Write your *own* instruction, point it at any synthetic report, and see what the
model does. (Run Step 6 above once first so the model is awake.)


In [ ]:
# ✏️ EDIT ME — write any instruction and run.  Synthetic data only.
from utils.ollama_client import chat, DEFAULT_HOST

MODEL = "llama3.2"

INSTRUCTION = "In one sentence, state any follow-up imaging the report recommends."

REPORT = """SYNTHETIC: 8 mm right lower lobe nodule, new since prior.
Recommend short-interval follow-up CT in 3 months."""

print(chat(f"{INSTRUCTION}\n\nREPORT:\n{REPORT}",
           model=MODEL, host=DEFAULT_HOST, temperature=0.0))
